# Expressions and contexts

In [2]:
import polars as pl
from datetime import date

## Expressions

In [5]:
bmi_expr = pl.col("weight") / (pl.col("height") ** 2)

bmi_expr

<Expr ['[(col("weight")) / (col("heigh…'] at 0x10F582A50>

## Contexts

In [ ]:
df = pl.DataFrame(
    {
        "name": ["Alice Archer", "Ben Brown", "Chloe Cooper", "Daniel Donovan"],
        "birthdate": [
            date(1997, 1, 10),
            date(1985, 2, 15),
            date(1983, 3, 22),
            date(1981, 4, 30),
        ],
        "weight": [57.9, 72.5, 53.6, 83.1], 
        "height": [1.56, 1.77, 1.65, 1.75], 
    }
)

df

name,birthdate,weight,height
str,date,f64,f64
"""Alice Archer""",1997-01-10,57.9,1.56
"""Ben Brown""",1985-02-15,72.5,1.77
"""Chloe Cooper""",1983-03-22,53.6,1.65
"""Daniel Donovan""",1981-04-30,83.1,1.75


### Select

In [16]:
result = df.select(
    bmi_expr.round(2).alias("bmi"),
    bmi_expr.mean().round(2).alias("avg_bmi"),
    pl.lit(25).alias("ideal_max_bmi"),
    pl.lit("just a test").alias("just a test"),
)

result

bmi,avg_bmi,ideal_max_bmi,just a test
f64,f64,i32,str
23.79,23.44,25,"""just a test"""
23.14,23.44,25,"""just a test"""
19.69,23.44,25,"""just a test"""
27.13,23.44,25,"""just a test"""


In [17]:
result = df.select(deviation=(bmi_expr - bmi_expr.mean()) / bmi_expr.std())

result

deviation
f64
0.115645
-0.097471
-1.22912
1.210946


### With columns

In [19]:
result = df.with_columns(
    bmi=bmi_expr,
    avg_bmi=bmi_expr.mean(),
    ideal_max_bmi=25,
)

result

name,birthdate,weight,height,bmi,avg_bmi,ideal_max_bmi
str,date,f64,f64,f64,f64,i32
"""Alice Archer""",1997-01-10,57.9,1.56,23.791913,23.438973,25
"""Ben Brown""",1985-02-15,72.5,1.77,23.141498,23.438973,25
"""Chloe Cooper""",1983-03-22,53.6,1.65,19.687787,23.438973,25
"""Daniel Donovan""",1981-04-30,83.1,1.75,27.134694,23.438973,25


### Filter

In [21]:
result = df.filter(
    pl.col("birthdate").is_between(date(1982, 12, 31), date(1996, 1, 1)),
    pl.col("height") > 1.7,
    pl.col("weight") < 80,
)

result

name,birthdate,weight,height
str,date,f64,f64
"""Ben Brown""",1985-02-15,72.5,1.77


### Group by and aggregations

In [ ]:
result = (
    df
    .group_by(
        decade=pl.col("birthdate").dt.year() // 10 * 10,
    )
    .agg(pl.col("name"))
)

result

decade,name
i32,list[str]
1980,"[""Ben Brown"", ""Chloe Cooper"", ""Daniel Donovan""]"
1990,"[""Alice Archer""]"


In [33]:
result = (
    df
    .group_by(
        (pl.col("height") < 1.7).alias("is_short"),
        (pl.col("birthdate").dt.year() // 10 * 10).alias("decade"),
    ).agg(pl.col("name"))
    .select("name", "is_short", "decade")
)

result

name,is_short,decade
list[str],bool,i32
"[""Ben Brown"", ""Daniel Donovan""]",false,1980
"[""Alice Archer""]",true,1990
"[""Chloe Cooper""]",true,1980


In [35]:
result = (
    df
    .group_by(
        (pl.col("birthdate").dt.year() // 10 * 10).alias("decade"),
        (pl.col("height") < 1.7).alias("short?"),
    ).agg(
        pl.len(),
        pl.col("height").max().alias("tallest"),
        pl.col("weight", "height").mean().name.prefix("avg_"),
    )
)

result

decade,short?,len,tallest,avg_weight,avg_height
i32,bool,u32,f64,f64,f64
1980,false,2,1.77,77.8,1.76
1980,true,1,1.65,53.6,1.65
1990,true,1,1.56,57.9,1.56


## Expression expansion